# Step 05.1 — Prediction、Target、Loss

到目前为止，我们已经会搭一个会做 forward 的网络。但它还不会“学习”。

训练的第一步不是 gradient，而是先回答：

> 模型预测了什么？正确答案是什么？它错了多少？

这三个词分别是：prediction、target、loss。

## 1. Prediction 和 Target

假设正确答案是 `target = 10`，模型预测 `prediction = 7`。

最直接的误差：

```text
prediction - target = -3
```

但正负误差可能互相抵消，所以通常不会直接把 signed error 当最终 loss。

In [ ]:
import mlx.core as mx

prediction = mx.array(7.0)
target = mx.array(10.0)

error = prediction - target
print("prediction:", prediction)
print("target    :", target)
print("error     :", error)

## 2. Squared Error

最简单的 loss 之一：

```text
loss = (prediction - target)^2
```

这里 `(7-10)^2 = 9`。

平方的两个直观作用：

1. 正负误差都变成非负；
2. 错得越远，惩罚增长更快。

In [ ]:
loss = mx.square(prediction - target)
mx.eval(loss)
print("loss:", loss)

## 3. 多个样本为什么要 mean？

例如：

```text
predictions = [7, 12, 9]
targets     = [10,10,10]
```

逐样本平方误差：`[9,4,1]`。

取平均：

```text
(9+4+1)/3 = 14/3
```

最终得到一个 scalar loss。

In [ ]:
predictions = mx.array([7.0, 12.0, 9.0])
targets = mx.array([10.0, 10.0, 10.0])

per_sample_loss = mx.square(predictions - targets)
mean_loss = mx.mean(per_sample_loss)

mx.eval(per_sample_loss, mean_loss)

print("per-sample loss:", per_sample_loss)
print("mean loss      :", mean_loss)

为什么常希望最后是 scalar？

因为下一步要研究：

> 如果某个 parameter 稍微改变，整体 loss 会怎么变？

这就是导数和 gradient 要解决的问题。

## 4. Loss function 在训练流程里的位置

```text
input
  ↓
model(parameters)
  ↓
prediction
  + target
  ↓
loss
```

模型负责产生 prediction；训练数据提供 target；loss function 负责比较两者。

loss 不是模型“自己知道正确答案”，而是外部训练目标。

## 5. 和 MiniGPT 的关系

这里使用 squared error，只为了把 `prediction → target → loss` 讲清楚。

语言模型预测的是“下一个 token 属于哪个类别”，最终会使用 Cross Entropy Loss，而不是 squared error。

现在只保留训练骨架：

```text
model(parameters) -> prediction -> compare with target -> scalar loss
```

### 5.1 结论

- prediction：模型输出；
- target：正确答案；
- loss：衡量预测与答案差距的训练目标；
- 多样本 loss 通常经过 reduction 变成 scalar；
- 下一节 5.2：导数——怎么知道 parameter 应该朝哪个方向改，才能让 loss 下降。